In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import matplotlib.dates as mdates
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb  # <--- The New Import

In [2]:
data = "/content/Data_Glico.xlsx - Data (1).csv"
df = pd.read_csv(data, thousands=',')


df = df.sort_values("Period Start").reset_index(drop=True)

print(f"Dataset loader: {len(df)} quaraters of data")
display(df)

Dataset loader: 50 quaraters of data


,Period Start,Period End,Revenue,COGS,Operating Income,Net Profit,Wheat Price,Sugar Price,CPI_YoY,Exchange_Rate,Income taxes - current,Income taxes - deferred,Total income taxes,Cash and Deposits,Total Assets,Population Youth,Snack Exp
0,1/1/2014,31/3/2014,69197,39377,-637,1251,249.46,370.97,-0.7,95.09,"5,581","2,664",8246,"15,262","189,937",16.6,76807.7
1,1/1/2015,31/3/2015,75198,41690,1613,6752,205.82,311.70,0.9,96.08,"4,047",-127,3920,"22,546","209,682",16.3,83027.0
2,1/1/2016,31/3/2016,76959,42116,249,-4422,162.65,316.26,1.6,90.20,"4,812",412,5225,"31,986","232,608",16.1,83472.0
3,1/1/2017,31/3/2017,82176,43928,1540,3993,143.65,433.60,3.4,91.57,"4,747",48,4795,"77,066","281,632",15.8,83087.0
4,1/1/2018,31/3/2018,81221,43422,1166,-725,169.87,296.73,3.3,91.26,"3,860",634,4495,"81,288","295,127",15.6,83916.0
5,1/1/2019,31/3/2019,78569,41579,290,-409,177.09,279.87,2.4,81.88,"5,580",246,5827,"75,323","294,523",15.4,87469.0
6,1/1/2020,31/3/2020,77999,39868,3465,-1815,173.78,301.61,0.6,83.74,"3,854",161,4016,"65,671","294,175",15.2,85534.0
7,1/1/2021,31/3/2021,74293,38720,4088,10103,236.21,358.05,0.2,83.37,"3,925",497,4422,"57,422","297,011",14.9,88195.0
8,1/1/2022,31/3/2022,65420,40486,4076,8856,353.75,409.85,0.2,85.86,"3,988",488,4477,"54,209","301,747",14.5,94373.0
9,1/1/2023,31/3/2023,71075,44761,3900,6042,320.65,457.89,-0.3,93.92,"2,909",138,3048,"56,333","316,960",14.2,99520.0


In [3]:
df["Period Start"] = pd.to_datetime(df["Period Start"], dayfirst=True)
df["Period End"] = pd.to_datetime(df["Period End"], dayfirst=True)

In [4]:
df = df.sort_values("Period Start").reset_index(drop=True)

def create_features(data):
  df_feat = data.copy()

  # Feature 1: Seasonality (Quarter 1-4)
  # Divide Quarters
  df_feat["Quarter"] = df_feat["Period Start"].dt.quarter

  # Feature 2: Lags (Past Performance)
  # The best predictor of tomorrow is often todat (Lag 1) or this time last year
  for col in ["Revenue", "Operating Income", "Net Profit"]:
    df_feat[f"{col}_Lag1"] = df_feat[col].shift(1) # Previous Quarter
    df_feat[f"{col}_Lag4"] = df_feat[col].shift(4) # Same Quarter Last Year

  # B. MACRO LAGS (Your Request: "Known at time t")
    # We shift these so we predict T using data from T-1
  macros = ['Wheat Price', 'Sugar Price', 'CPI_YoY', 'Exchange_Rate', "Population Youth", "Snack Exp", "COGS"]
  for col in macros:
      df_feat[f'{col}_Lag1'] = df_feat[col].shift(1)

  # Drop the first 4 rows because they now have NaNs (no history for them)
  df_feat = df_feat.dropna().reset_index(drop=True)

  return df_feat

df_model = create_features(df)


In [15]:
display(df_model)

,Period Start,Period End,Revenue,COGS,Operating Income,Net Profit,Wheat Price,Sugar Price,CPI_YoY,Exchange_Rate,...,Operating Income_Lag4,Net Profit_Lag1,Net Profit_Lag4,Wheat Price_Lag1,Sugar Price_Lag1,CPI_YoY_Lag1,Exchange_Rate_Lag1,Population Youth_Lag1,Snack Exp_Lag1,COGS_Lag1
0,2014-04-01,2014-06-30,78853,43611,4607,11584,275.32,401.95,-0.3,92.24,...,3813.0,1251.0,4927.0,249.46,370.97,-0.7,95.09,16.6,76807.7,39377.0
1,2014-07-01,2014-09-30,86872,48657,4876,4853,226.29,389.48,0.2,96.27,...,5971.0,11584.0,5518.0,275.32,401.95,-0.3,92.24,16.6,76807.7,43611.0
2,2014-10-01,2014-12-31,78470,43824,3151,5299,220.52,347.94,0.7,94.38,...,2498.0,4853.0,3818.0,226.29,389.48,0.2,96.27,16.6,76807.7,48657.0
3,2015-01-01,2015-03-31,75198,41690,1613,6752,205.82,311.70,0.9,96.08,...,-637.0,5299.0,1251.0,220.52,347.94,0.7,94.38,16.6,76807.7,43824.0
4,2015-04-01,2015-06-30,85167,45998,5739,4378,196.33,277.25,1.1,94.58,...,4607.0,6752.0,11584.0,205.82,311.70,0.9,96.08,16.3,83027.0,41690.0
5,2015-07-01,2015-09-30,95019,51293,6855,3483,178.93,254.93,1.1,95.01,...,4876.0,4378.0,4853.0,196.33,277.25,1.1,94.58,16.3,83027.0,45998.0
6,2015-10-01,2015-12-31,81292,44497,4267,3972,162.38,321.49,1.5,93.35,...,3151.0,3483.0,5299.0,178.93,254.93,1.1,95.01,16.3,83027.0,51293.0
7,2016-01-01,2016-03-31,76959,42116,249,-4422,162.65,316.26,1.6,90.20,...,1613.0,3972.0,6752.0,162.38,321.49,1.5,93.35,16.3,83027.0,44497.0
8,2016-04-01,2016-06-30,88747,47477,7840,4776,159.21,374.93,1.4,90.17,...,5739.0,-4422.0,4378.0,162.65,316.26,1.6,90.20,16.1,83472.0,42116.0
9,2016-07-01,2016-09-30,97495,51830,8583,4659,128.29,448.67,1.5,92.04,...,6855.0,4776.0,3483.0,159.21,374.93,1.4,90.17,16.1,83472.0,47477.0


In [5]:
features_rev = ["Quarter", "Wheat Price_Lag1", "Sugar Price_Lag1", "CPI_YoY_Lag1", "Exchange_Rate_Lag1", "Revenue_Lag1", "Revenue_Lag4", "Population Youth_Lag1", "Snack Exp_Lag1"]
target_rev = "Revenue"

In [6]:
X_rev = df_model[features_rev]
y_rev = df_model[target_rev]

In [7]:
model_rev = xgb.XGBRegressor(
        n_estimators=200,
        learning_rate=0.01,   # 'eta' in XGBoost terms, controls learning speed
        max_depth=3,          # Keep trees simple to avoid overfitting
        random_state=42,
        objective='reg:squarederror', # Standard regression objective
        n_jobs=-1,            # Use all CPU cores for speed
        subsample=0.8,        # (Optional) Randomly sample 80% of rows per tree to prevent overfitting
        colsample_bytree=0.8  # (Optional) Randomly sample 80% of features per tree
    )

model_rev.fit(X_rev, y_rev)

# Now we generate the future prediction with confidence

last_row = df_model.iloc[-1]
row_minus_3 = df_model.iloc[-3] # Value from 3 quarters ago (needed for Lag4 of next Q)

future_inputs = pd.DataFrame([{
    'Quarter': (last_row["Period Start"].month // 3) % 4 + 1,

    # Macro Lags (Shifted forward: Lag1 for next Q is Current Value of last Q)
    'Wheat Price_Lag1': last_row['Wheat Price'],       # Use most recent actual price
    'Sugar Price_Lag1': last_row['Sugar Price'],       # Use most recent actual price
    'CPI_YoY_Lag1': last_row['CPI_YoY'],               # Use most recent actual
    'Exchange_Rate_Lag1': last_row['Exchange_Rate'],   # Use most recent actual

    # Operating Income Lags (Shifted forward)
    'Revenue_Lag1': last_row['Revenue'],     # Last quarter's actual
    'Revenue_Lag4': row_minus_3['Revenue'],  # 3 quarters ago (becomes 4 ago next Q)

    # Other Lags
    'Population Youth_Lag1': last_row['Population Youth'],
    'Snack Exp_Lag1': last_row['Snack Exp']
}])

predicted_revenue = model_rev.predict(future_inputs)[0]

print("\n--- FINAL FORECAST REPORT ---")
print(f"Last Actual Revenue: {last_row['Revenue']:,.0f}")
print(f"Predicted Revenue for Next Quarter: {predicted_revenue:,.0f}")


--- FINAL FORECAST REPORT ---
Last Actual Revenue: 100,241
Predicted Revenue for Next Quarter: 86,087


In [8]:
features_op = [
    "Quarter",
    "Wheat Price_Lag1", "Sugar Price_Lag1", "CPI_YoY_Lag1", "Exchange_Rate_Lag1",
    "Operating Income_Lag1", "Operating Income_Lag4",
    "Population Youth_Lag1", "Snack Exp_Lag1","COGS_Lag1",
    "Revenue"  # <--- The Critical Feature
]
target_op = "Operating Income"

In [9]:
X_op = df_model[features_op]
y_op = df_model[target_op]

In [10]:
model_op = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.01,
    max_depth=3,
    random_state=42,
    objective='reg:squarederror',
    n_jobs=-1
)

model_op.fit(X_op, y_op)
print("Operating Income Model Trained.")

# --- 2. Construct Future Inputs (The Chain) ---
last_row = df_model.iloc[-1]
row_minus_3 = df_model.iloc[-3] # Value from 3 quarters ago (needed for Lag4 of next Q)

future_inputs_op = pd.DataFrame([{
    'Quarter': (last_row["Period Start"].month // 3) % 4 + 1,

    # Macro Lags (Shifted forward: Lag1 for next Q is Current Value of last Q)
    'Wheat Price_Lag1': last_row['Wheat Price'],       # Use most recent actual price
    'Sugar Price_Lag1': last_row['Sugar Price'],       # Use most recent actual price
    'CPI_YoY_Lag1': last_row['CPI_YoY'],               # Use most recent actual
    'Exchange_Rate_Lag1': last_row['Exchange_Rate'],   # Use most recent actual

    # Operating Income Lags (Shifted forward)
    'Operating Income_Lag1': last_row['Operating Income'],     # Last quarter's actual
    'Operating Income_Lag4': row_minus_3['Operating Income'],  # 3 quarters ago (becomes 4 ago next Q)

    # Other Lags
    'Population Youth_Lag1': last_row['Population Youth'],
    'Snack Exp_Lag1': last_row['Snack Exp'],
    'COGS_Lag1': last_row['COGS'],


    # *** THE CHAIN LINK ***
    'Revenue': predicted_revenue  # <--- Inject the Revenue prediction here!
}])

# --- 3. Predict ---
predicted_op_income = model_op.predict(future_inputs_op)[0]

print("\n--- OPERATING INCOME FORECAST ---")
print(f"Inputted Revenue (Predicted): {predicted_revenue:,.0f}")
print(f"Predicted Operating Income:   {predicted_op_income:,.0f}")

Operating Income Model Trained.

--- OPERATING INCOME FORECAST ---
Inputted Revenue (Predicted): 86,087
Predicted Operating Income:   1,093


In [11]:
features_np = [
    "Quarter",
    "Wheat Price_Lag1", "Sugar Price_Lag1", "CPI_YoY_Lag1", "Exchange_Rate_Lag1",
    "Net Profit_Lag1", "Net Profit_Lag4",
    "Population Youth_Lag1", "Snack Exp_Lag1","COGS_Lag1",
    "Revenue",           # <--- Chain Link 1
    "Operating Income"   # <--- Chain Link 2
]
target_np = "Net Profit"

X_np = df_model[features_np]
y_np = df_model[target_np]

In [12]:
model_np = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.01,
    max_depth=3,
    random_state=42,
    objective='reg:squarederror',
    n_jobs=-1
)

model_np.fit(X_np, y_np)
print("Net Profit Model Trained.")

# --- 2. Construct Future Inputs (The Chain) ---
last_row = df_model.iloc[-1]
row_minus_3 = df_model.iloc[-3] # Value from 3 quarters ago (becomes Lag4 for next Q)

future_inputs_np = pd.DataFrame([{
    'Quarter': (last_row["Period Start"].month // 3) % 4 + 1,

    # Macro Lags (Assumed Constant from previous step logic)
    'Wheat Price_Lag1': last_row['Wheat Price_Lag1'],
    'Sugar Price_Lag1': last_row['Sugar Price_Lag1'],
    'CPI_YoY_Lag1': last_row['CPI_YoY_Lag1'],
    'Exchange_Rate_Lag1': last_row['Exchange_Rate_Lag1'],

    # Net Profit Lags (Shifted forward)
    'Net Profit_Lag1': last_row['Net Profit'],    # Last quarter's actual
    'Net Profit_Lag4': row_minus_3['Net Profit'], # 3 quarters ago (becomes 4 ago next Q)

    # Other Lags
    'Population Youth_Lag1': last_row['Population Youth_Lag1'],
    'Snack Exp_Lag1': last_row['Snack Exp_Lag1'],
    'COGS_Lag1': last_row['COGS'],


    # *** THE DOUBLE CHAIN LINK ***
    'Revenue': predicted_revenue,          # <--- Inject Revenue Prediction
    'Operating Income': predicted_op_income # <--- Inject Op Income Prediction
}])

# --- 3. Predict ---
predicted_net_profit = model_np.predict(future_inputs_np)[0]

print("\n--- FINAL FORECAST SUMMARY ---")
print(f"1. Predicted Revenue:          {predicted_revenue:,.0f}")
print(f"2. Predicted Operating Income: {predicted_op_income:,.0f}")
print(f"3. Predicted Net Profit:       {predicted_net_profit:,.0f}")

Net Profit Model Trained.

--- FINAL FORECAST SUMMARY ---
1. Predicted Revenue:          86,087
2. Predicted Operating Income: 1,093
3. Predicted Net Profit:       -197


Utilizing the rolling forecast loop

In [13]:
import pandas as pd
import numpy as np
import xgboost as xgb

# [Models 'model_rev', 'model_op', 'model_np' assumed ready]
# [Feature lists 'features_rev', 'features_op', 'features_np' assumed ready]

# 2. Define Simulation
last_known_date = df['Period Start'].max()
target_date = pd.Timestamp("2026-10-01")

# Create History Buffer
history = df.copy()

print(f"--- STARTING MULTI-STEP FORECAST ---")
print(f"From: {last_known_date.date()}")
print(f"To:   {target_date.date()}\n")
print(f"{'Date':<12} | {'Revenue':>12} | {'Op Income':>12} | {'Net Profit':>12}")
print("-" * 60)

# 3. The Loop
current_date = last_known_date + pd.DateOffset(months=3)

while current_date <= target_date:
    # A. Get Lags
    row_lag1 = history.iloc[-1] # Previous Quarter
    row_lag4 = history.iloc[-4] # Previous Year

    # B. Construct Inputs
    input_row = {
        'Quarter': (current_date.month // 3) % 4 + 1,

        # --- MACRO LAGS (Carried Forward) ---
        'Wheat Price_Lag1': row_lag1['Wheat Price'],
        'Sugar Price_Lag1': row_lag1['Sugar Price'],
        'CPI_YoY_Lag1': row_lag1['CPI_YoY'],
        'Exchange_Rate_Lag1': row_lag1['Exchange_Rate'],

        # --- THE MISSING PIECES (Carried Forward) ---
        'Population Youth_Lag1': row_lag1['Population Youth'],
        'Snack Exp_Lag1': row_lag1['Snack Exp'],
        'COGS_Lag1': last_row['COGS'],

        # --- PERFORMANCE LAGS ---
        'Revenue_Lag1': row_lag1['Revenue'],
        'Revenue_Lag4': row_lag4['Revenue'],
        'Operating Income_Lag1': row_lag1['Operating Income'],
        'Operating Income_Lag4': row_lag4['Operating Income'],
        'Net Profit_Lag1': row_lag1['Net Profit'],
        'Net Profit_Lag4': row_lag4['Net Profit']
    }

    # C. Predict REVENUE
    input_df_rev = pd.DataFrame([input_row])[features_rev]
    pred_revenue = model_rev.predict(input_df_rev)[0]

    # D. Predict OP INCOME (Inject Revenue)
    input_row['Revenue'] = pred_revenue
    input_df_op = pd.DataFrame([input_row])[features_op]
    pred_op = model_op.predict(input_df_op)[0]

    # E. Predict NET PROFIT (Inject Op Income)
    input_row['Operating Income'] = pred_op
    input_df_np = pd.DataFrame([input_row])[features_np]
    pred_np = model_np.predict(input_df_np)[0]

    # F. Print
    print(f"{current_date.date()} | {pred_revenue:,.0f} | {pred_op:,.0f} | {pred_np:,.0f}")

    # G. Append to History
    new_row = {
        'Period Start': current_date,
        'Revenue': pred_revenue,
        'Operating Income': pred_op,
        'Net Profit': pred_np,

        # --- CRITICAL: Save these so they become "Lag1" for the next loop ---
        'Wheat Price': row_lag1['Wheat Price'],
        'Sugar Price': row_lag1['Sugar Price'],
        'CPI_YoY': row_lag1['CPI_YoY'],
        'Exchange_Rate': row_lag1['Exchange_Rate'], # Check your column name (Rate vs Rate_)
        'Population Youth': row_lag1['Population Youth'], # <--- Added
        'Snack Exp': row_lag1['Snack Exp'] ,               # <--- Added
        'COGS': last_row['COGS'],

    }

    # Use pandas.concat instead of append
    history = pd.concat([history, pd.DataFrame([new_row])], ignore_index=True)

    current_date = current_date + pd.DateOffset(months=3)

print("-" * 60)
print("Forecast Complete.")

--- STARTING MULTI-STEP FORECAST ---
From: 2025-07-01
To:   2026-10-01

Date         |      Revenue |    Op Income |   Net Profit
------------------------------------------------------------
2025-10-01 | 91,704 | 1,061 | -197
2026-01-01 | 79,606 | 1,014 | -13
2026-04-01 | 88,299 | 982 | -13
2026-07-01 | 93,950 | 6,614 | 3,477
2026-10-01 | 92,004 | 1,061 | -13
------------------------------------------------------------
Forecast Complete.


In [19]:
# ==========================================
# CALCULATION: AGGREGATE FY FROM 'HISTORY'
# ==========================================

# 1. Extract Year from the Date column in your 'history' dataframe
# (Make sure 'history' is the dataframe you used in the loop)
history['Year'] = pd.to_datetime(history['Period Start']).dt.year

# 2. Filter for only the forecast years (2025 and 2026)
df_fy_forecast = history[history['Year'].isin([2025, 2026])].copy()

# 3. Group by Year and Sum the quarterly values
df_fy_totals = df_fy_forecast.groupby('Year')[['Revenue', 'Operating Income', 'Net Profit']].sum().reset_index()

# 4. Display the Final Table
print("\n--- FINAL FULL YEAR (FY) PROJECTIONS (Millions JPY) ---")
print(df_fy_totals.to_string(index=False, float_format="{:,.0f}".format))

# 5. Calculate & Print Growth Rates
if len(df_fy_totals) == 2:
    p_2025 = df_fy_totals[df_fy_totals["Year"] == 2025].iloc[0]
    p_2026 = df_fy_totals[df_fy_totals["Year"] == 2026].iloc[0]

    print("\n--- YoY Growth (2026 vs 2025) ---")
    for col in ["Revenue", "Operating Income", "Net Profit"]:
        val_25 = p_2025[col]
        val_26 = p_2026[col]
        growth = ((val_26 - val_25) / val_25) * 100
        print(f"{col:<20} : {growth:>6.2f}%")


--- FINAL FULL YEAR (FY) PROJECTIONS (Millions JPY) ---
 Year  Revenue  Operating Income  Net Profit
 2025  356,440            10,081       5,553
 2026  353,859             9,671       3,438

--- YoY Growth (2026 vs 2025) ---
Revenue              :  -0.72%
Operating Income     :  -4.06%
Net Profit           : -38.09%
